In [ ]:
%cd bleurt
!pip install .

In [ ]:
!python -m unittest bleurt.score_test
!python -m unittest bleurt.score_not_eager_test
!python -m unittest bleurt.finetune_test
!python -m unittest bleurt.score_files_test

In [ ]:
import os
import logging

# Set TensorFlow to only display WARNING messages or higher
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
logging.getLogger('tensorflow').setLevel(logging.WARNING)
from rouge_score import rouge_scorer
import pandas as pd
from bleurt import score

In [ ]:
dataset= "new_ds_test.csv"
checkpoint = "bleurt/bleurt/test_checkpoint"
df = pd.read_csv(dataset)
def calculate_rouge(reference, generated):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference, generated)
    return scores
def calculate_bleurt(reference, generated):
    bleurt_scorer = score.BleurtScorer(checkpoint)
    score_value = bleurt_scorer.score(references=[reference], candidates=[generated])[0]
    return score_value

In [ ]:
df['rouge1_f'] = 0.0
df['rouge2_f'] = 0.0
df['rougeL_f'] = 0.0
df['Bleurt Score'] = 0.0

for index, row in df.iterrows():
    print(f"Processing row {index}")
    #rouge
    scores = calculate_rouge(row['Output'], row['Generated'])
    df.at[index, 'rouge1_f'] = scores['rouge1'].fmeasure
    df.at[index, 'rouge2_f'] = scores['rouge2'].fmeasure
    df.at[index, 'rougeL_f'] = scores['rougeL'].fmeasure
    #bleurt
    bleurt_score = calculate_bleurt(row['Output'], row['Generated'])
    df.at[index, 'Bleurt Score'] = bleurt_score
df['Rouge Score'] = (df['rouge1_f'] + df['rouge2_f'] + df['rougeL_f']) / 3

df.to_csv('dataset_scores.csv', index=False)